In [ ]:
# ============================================================
# AI CUP ESG 永續承諾驗證競賽
# BERT 穩定版完整程式
#
# 重要：
# 1. 請先「執行階段 → 重新啟動執行階段」
# 2. 然後只跑這一格
# 3. 最後會產生 submission_bert_stable.csv
# ============================================================

# ============================================================
# 0. 安裝穩定版本
# ============================================================

!pip -q uninstall -y peft
!pip -q install -U "transformers==4.41.2" "tokenizers==0.19.1" "accelerate==0.30.1"

# ============================================================
# 1. 匯入套件
# ============================================================

import os
import gc
import random
import numpy as np
import pandas as pd
import torch

from google.colab import files

from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW

from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import f1_score, classification_report

from transformers import (
    BertTokenizerFast,
    BertForSequenceClassification,
    get_linear_schedule_with_warmup
)

# ============================================================
# 2. 基本設定
# ============================================================

SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("目前裝置：", device)

if str(device) != "cuda":
    print("警告：你現在不是 GPU，會跑很慢。")
    print("請到：執行階段 → 變更執行階段類型 → T4 GPU")

# ============================================================
# 3. 找檔案
# ============================================================

def find_file(keyword, ext=".csv"):
    for fname in os.listdir("/content"):
        if keyword in fname and fname.endswith(ext):
            return "/content/" + fname
    return None

TRAIN_PATH = find_file("train_1000", ".csv")
VAL_PATH = find_file("val_1000", ".csv")
TEST_PATH = find_file("test_2000", ".csv")

if TRAIN_PATH is None or VAL_PATH is None or TEST_PATH is None:
    print("找不到完整 CSV，請上傳這三個 CSV：")
    print("1. vpesg4k_train_1000 V1.csv")
    print("2. vpesg4k_val_1000.csv")
    print("3. vpesg4k_test_2000.csv")
    uploaded = files.upload()

    TRAIN_PATH = find_file("train_1000", ".csv")
    VAL_PATH = find_file("val_1000", ".csv")
    TEST_PATH = find_file("test_2000", ".csv")

print("TRAIN_PATH =", TRAIN_PATH)
print("VAL_PATH   =", VAL_PATH)
print("TEST_PATH  =", TEST_PATH)

# ============================================================
# 4. 讀取資料
# ============================================================

train_df = pd.read_csv(TRAIN_PATH, keep_default_na=False)
val_df = pd.read_csv(VAL_PATH, keep_default_na=False)
test_df = pd.read_csv(TEST_PATH, keep_default_na=False)

print("\n資料大小：")
print("train:", train_df.shape)
print("val:", val_df.shape)
print("test:", test_df.shape)

print("\ntrain 欄位：", train_df.columns.tolist())
print("val 欄位：", val_df.columns.tolist())
print("test 欄位：", test_df.columns.tolist())

# ============================================================
# 5. 標籤欄位與官方權重
# ============================================================

TARGET_COLS = [
    "promise_status",
    "evidence_status",
    "evidence_quality",
    "verification_timeline"
]

WEIGHTS = {
    "promise_status": 0.20,
    "evidence_status": 0.30,
    "evidence_quality": 0.35,
    "verification_timeline": 0.15
}

# ============================================================
# 6. 標籤修正
# ============================================================

def normalize_labels(df):
    df = df.copy()

    if "verification_timeline" in df.columns:
        df["verification_timeline"] = df["verification_timeline"].replace({
            "longer_than_5_years": "more_than_5_years"
        })

    return df

train_df = normalize_labels(train_df)
val_df = normalize_labels(val_df)

# ============================================================
# 7. 建立輸入文字
#
# 注意：
# test 沒有 esg_type，所以這版不使用 esg_type
# 避免模型在訓練時依賴 test 沒有的欄位
# ============================================================

def build_text(df):
    df = df.copy()

    for col in ["data", "company", "ticker", "page_number"]:
        if col not in df.columns:
            df[col] = ""

    text = (
        "公司：" + df["company"].astype(str) +
        "。股票代號：" + df["ticker"].astype(str) +
        "。頁碼：" + df["page_number"].astype(str) +
        "。內容：" + df["data"].astype(str)
    )

    return text.tolist()

X_train_text = build_text(train_df)
X_val_text = build_text(val_df)
X_test_text = build_text(test_df)

# ============================================================
# 8. 模型設定
#
# 這版用最穩的 bert-base-chinese
# 如果要之後再拚更高，可以把 MODEL_NAME 換 hfl/chinese-macbert-base
# ============================================================

MODEL_NAME = "bert-base-chinese"

MAX_LENGTH = 512
BATCH_SIZE = 8

# 驗證階段訓練回合
EPOCHS_VAL = 3

# 最終 train + val 合併訓練回合
EPOCHS_FINAL = 3

LR = 2e-5

tokenizer = BertTokenizerFast.from_pretrained(MODEL_NAME)

# ============================================================
# 9. Dataset
# ============================================================

class ESGDataset(Dataset):
    def __init__(self, texts, labels=None):
        self.texts = texts
        self.labels = labels

        self.encodings = tokenizer(
            texts,
            truncation=True,
            padding=True,
            max_length=MAX_LENGTH
        )

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        item = {
            "input_ids": torch.tensor(self.encodings["input_ids"][idx], dtype=torch.long),
            "attention_mask": torch.tensor(self.encodings["attention_mask"][idx], dtype=torch.long)
        }

        if self.labels is not None:
            item["labels"] = torch.tensor(self.labels[idx], dtype=torch.long)

        return item

# ============================================================
# 10. 類別權重
# ============================================================

def make_class_weights(y, num_labels):
    counts = np.bincount(y, minlength=num_labels)
    counts = np.maximum(counts, 1)

    weights = len(y) / (num_labels * counts)
    weights = np.sqrt(weights)
    weights = np.clip(weights, 0.5, 5.0)

    return torch.tensor(weights, dtype=torch.float).to(device)

# ============================================================
# 11. 預測函式
# ============================================================

def predict_with_model(model, label_encoder, dataloader):
    model.eval()

    all_pred_ids = []

    with torch.no_grad():
        for batch in dataloader:
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)

            outputs = model(
                input_ids=input_ids,
                attention_mask=attention_mask
            )

            logits = outputs.logits
            pred_ids = torch.argmax(logits, dim=1)

            all_pred_ids.extend(pred_ids.cpu().numpy().tolist())

    pred_texts = label_encoder.inverse_transform(all_pred_ids)

    return pred_texts

# ============================================================
# 12. 訓練單一任務
# ============================================================

def train_single_task(
    target_col,
    train_texts,
    train_label_texts,
    val_texts=None,
    val_label_texts=None,
    epochs=3
):
    print("\n" + "=" * 90)
    print("開始訓練任務：", target_col)
    print("=" * 90)

    label_encoder = LabelEncoder()
    y_train = label_encoder.fit_transform(train_label_texts)

    num_labels = len(label_encoder.classes_)

    print("標籤：", list(label_encoder.classes_))

    train_dataset = ESGDataset(train_texts, y_train)

    train_loader = DataLoader(
        train_dataset,
        batch_size=BATCH_SIZE,
        shuffle=True
    )

    val_loader = None

    if val_texts is not None and val_label_texts is not None:
        y_val = label_encoder.transform(val_label_texts)
        val_dataset = ESGDataset(val_texts, y_val)

        val_loader = DataLoader(
            val_dataset,
            batch_size=BATCH_SIZE * 2,
            shuffle=False
        )

    model = BertForSequenceClassification.from_pretrained(
        MODEL_NAME,
        num_labels=num_labels
    )

    model.to(device)

    optimizer = AdamW(
        model.parameters(),
        lr=LR,
        weight_decay=0.01
    )

    total_steps = len(train_loader) * epochs
    warmup_steps = int(total_steps * 0.1)

    scheduler = get_linear_schedule_with_warmup(
        optimizer,
        num_warmup_steps=warmup_steps,
        num_training_steps=total_steps
    )

    class_weights = make_class_weights(y_train, num_labels)
    loss_fn = torch.nn.CrossEntropyLoss(weight=class_weights)

    best_f1 = -1
    best_state_dict = None

    scaler = torch.cuda.amp.GradScaler(enabled=(str(device) == "cuda"))

    for epoch in range(epochs):
        model.train()

        total_loss = 0.0

        print(f"\nEpoch {epoch + 1}/{epochs}")

        for step, batch in enumerate(train_loader):
            optimizer.zero_grad()

            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels = batch["labels"].to(device)

            with torch.cuda.amp.autocast(enabled=(str(device) == "cuda")):
                outputs = model(
                    input_ids=input_ids,
                    attention_mask=attention_mask
                )

                logits = outputs.logits
                loss = loss_fn(logits, labels)

            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()

            scheduler.step()

            total_loss += loss.item()

            if (step + 1) % 30 == 0:
                avg_now = total_loss / (step + 1)
                print(f"step {step + 1}/{len(train_loader)} loss={avg_now:.4f}")

        avg_loss = total_loss / len(train_loader)
        print("平均 loss =", round(avg_loss, 4))

        if val_loader is not None:
            pred_texts = predict_with_model(model, label_encoder, val_loader)

            macro_f1 = f1_score(
                val_label_texts,
                pred_texts,
                average="macro",
                zero_division=0
            )

            print("Val Macro-F1 =", round(macro_f1, 4))

            if macro_f1 > best_f1:
                best_f1 = macro_f1
                best_state_dict = {
                    k: v.cpu().clone()
                    for k, v in model.state_dict().items()
                }

    if best_state_dict is not None:
        model.load_state_dict(best_state_dict)
        model.to(device)

    final_val_pred = None

    if val_loader is not None:
        final_val_pred = predict_with_model(model, label_encoder, val_loader)

        print("\n任務：", target_col)
        print("最佳 Val Macro-F1 =", round(best_f1, 4))
        print(classification_report(val_label_texts, final_val_pred, zero_division=0))

    return model, label_encoder, final_val_pred

# ============================================================
# 13. 規則修正
# ============================================================

def apply_rules(pred_df):
    pred_df = pred_df.copy()

    pred_df["verification_timeline"] = pred_df["verification_timeline"].replace({
        "longer_than_5_years": "more_than_5_years"
    })

    # promise_status = No，後面一定 N/A
    mask_no_promise = pred_df["promise_status"] == "No"
    pred_df.loc[mask_no_promise, "verification_timeline"] = "N/A"
    pred_df.loc[mask_no_promise, "evidence_status"] = "N/A"
    pred_df.loc[mask_no_promise, "evidence_quality"] = "N/A"

    # promise_status = Yes，timeline 不應該是 N/A
    mask_yes_timeline_na = (
        (pred_df["promise_status"] == "Yes") &
        (pred_df["verification_timeline"] == "N/A")
    )
    pred_df.loc[mask_yes_timeline_na, "verification_timeline"] = "already"

    # promise_status = Yes，evidence_status 不應該是 N/A
    mask_yes_evidence_na = (
        (pred_df["promise_status"] == "Yes") &
        (pred_df["evidence_status"] == "N/A")
    )
    pred_df.loc[mask_yes_evidence_na, "evidence_status"] = "Yes"

    # evidence_status = No 或 N/A，quality 一定 N/A
    mask_no_evidence = pred_df["evidence_status"].isin(["No", "N/A"])
    pred_df.loc[mask_no_evidence, "evidence_quality"] = "N/A"

    # evidence_status = Yes，但 quality = N/A，改 Clear
    mask_yes_quality_na = (
        (pred_df["evidence_status"] == "Yes") &
        (pred_df["evidence_quality"] == "N/A")
    )
    pred_df.loc[mask_yes_quality_na, "evidence_quality"] = "Clear"

    return pred_df

# ============================================================
# 14. 計算官方加權分數
# ============================================================

def compute_weighted_score(true_df, pred_df):
    result = {}

    for col in TARGET_COLS:
        result[col] = f1_score(
            true_df[col],
            pred_df[col],
            average="macro",
            zero_division=0
        )

    weighted_score = (
        result["promise_status"] * WEIGHTS["promise_status"] +
        result["evidence_status"] * WEIGHTS["evidence_status"] +
        result["evidence_quality"] * WEIGHTS["evidence_quality"] +
        result["verification_timeline"] * WEIGHTS["verification_timeline"]
    )

    return result, weighted_score

# ============================================================
# 15. 第一階段：train 訓練，val 驗證
# ============================================================

print("\n" + "#" * 90)
print("第一階段：train 訓練，val 驗證")
print("#" * 90)

val_pred = pd.DataFrame()
val_pred["id"] = val_df["id"]

for target in TARGET_COLS:
    model, encoder, pred = train_single_task(
        target_col=target,
        train_texts=X_train_text,
        train_label_texts=train_df[target].tolist(),
        val_texts=X_val_text,
        val_label_texts=val_df[target].tolist(),
        epochs=EPOCHS_VAL
    )

    val_pred[target] = pred

    del model
    del encoder
    gc.collect()

    if torch.cuda.is_available():
        torch.cuda.empty_cache()

val_pred = normalize_labels(val_pred)
val_pred_rule = apply_rules(val_pred)

raw_scores, raw_weighted = compute_weighted_score(val_df, val_pred)
rule_scores, rule_weighted = compute_weighted_score(val_df, val_pred_rule)

print("\n" + "=" * 90)
print("BERT 未套規則驗證集分數")
print("=" * 90)

for k, v in raw_scores.items():
    print(k, "=", round(v, 4))

print("weighted_score =", round(raw_weighted, 4))

print("\n" + "=" * 90)
print("BERT 套規則後驗證集分數")
print("=" * 90)

for k, v in rule_scores.items():
    print(k, "=", round(v, 4))

print("weighted_score =", round(rule_weighted, 4))

val_pred.to_csv("/content/val_prediction_bert_raw.csv", index=False, encoding="utf-8-sig")
val_pred_rule.to_csv("/content/val_prediction_bert_rule.csv", index=False, encoding="utf-8-sig")

# ============================================================
# 16. 第二階段：合併 train + val，訓練最終模型並預測 test
# ============================================================

print("\n" + "#" * 90)
print("第二階段：合併 train + val，訓練最終模型並預測 test")
print("#" * 90)

full_df = pd.concat([train_df, val_df], ignore_index=True)
full_df = normalize_labels(full_df)

X_full_text = build_text(full_df)

test_dataset = ESGDataset(X_test_text, labels=None)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE * 2,
    shuffle=False
)

submission = pd.DataFrame()
submission["id"] = test_df["id"]

for target in TARGET_COLS:
    model, encoder, _ = train_single_task(
        target_col=target,
        train_texts=X_full_text,
        train_label_texts=full_df[target].tolist(),
        val_texts=None,
        val_label_texts=None,
        epochs=EPOCHS_FINAL
    )

    print("\n正在預測 test 欄位：", target)

    pred = predict_with_model(
        model=model,
        label_encoder=encoder,
        dataloader=test_loader
    )

    submission[target] = pred

    del model
    del encoder
    gc.collect()

    if torch.cuda.is_available():
        torch.cuda.empty_cache()

# ============================================================
# 17. 套規則並輸出 submission
# ============================================================

submission = normalize_labels(submission)
submission = apply_rules(submission)

submission = submission[
    [
        "id",
        "promise_status",
        "verification_timeline",
        "evidence_status",
        "evidence_quality"
    ]
]

SUBMISSION_PATH = "/content/submission_bert_stable.csv"

submission.to_csv(SUBMISSION_PATH, index=False, encoding="utf-8-sig")

print("\nsubmission 預覽：")
print(submission.head())

print("\nsubmission 大小：")
print(submission.shape)

print("\n各欄位標籤分布：")
for col in ["promise_status", "verification_timeline", "evidence_status", "evidence_quality"]:
    print("\n" + col)
    print(submission[col].value_counts())

print("\n已產生：", SUBMISSION_PATH)

files.download(SUBMISSION_PATH)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 302.6/302.6 kB 9.1 MB/s eta 0:00:00
目前裝置： cuda
TRAIN_PATH = /content/vpesg4k_train_1000 V1.csv
VAL_PATH   = /content/vpesg4k_val_1000.csv
TEST_PATH  = /content/vpesg4k_test_2000.csv

資料大小：
train: (1000, 14)
val: (1000, 14)
test: (2000, 7)

train 欄位： ['id', 'data', 'esg_type', 'promise_status', 'promise_string', 'verification_timeline', 'evidence_status', 'evidence_string', 'evidence_quality', 'company', 'ticker', 'page_number', 'pdf_url', 'company_source']
val 欄位： ['id', 'data', 'esg_type', 'promise_status', 'promise_string', 'verification_timeline', 'evidence_status', 'evidence_string', 'evidence_quality', 'company', 'ticker', 'page_number', 'pdf_url', 'company_source']
test 欄位： ['id', 'data', 'company', 'ticker', 'page_number', 'pdf_url', 'company_source']


tokenizer_config.json:   0%|          | 0.00/49.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


...


Epoch 1/3
step 30/250 loss=1.3108
step 60/250 loss=1.2387
step 90/250 loss=1.1519
step 120/250 loss=1.1199
step 150/250 loss=1.0910
step 180/250 loss=1.0637
step 210/250 loss=1.0475
step 240/250 loss=1.0321
平均 loss = 1.0221

Epoch 2/3
step 30/250 loss=0.8165
step 60/250 loss=0.8353
step 90/250 loss=0.8570
step 120/250 loss=0.8546
step 150/250 loss=0.8588
step 180/250 loss=0.8397
step 210/250 loss=0.8275
step 240/250 loss=0.8134
平均 loss = 0.8222

Epoch 3/3
step 30/250 loss=0.6494
step 60/250 loss=0.6729
step 90/250 loss=0.6398
step 120/250 loss=0.6343
step 150/250 loss=0.6407
step 180/250 loss=0.6417
step 210/250 loss=0.6373
step 240/250 loss=0.6448
平均 loss = 0.6444

正在預測 test 欄位： evidence_quality

開始訓練任務： verification_timeline
標籤： [np.str_('N/A'), np.str_('already'), np.str_('between_2_and_5_years'), np.str_('more_than_5_years'), np.str_('within_2_years')]


Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-chinese and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
/tmp/ipykernel_3930/3116446850.py:338: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler(enabled=(str(device) == "cuda"))
/tmp/ipykernel_3930/3116446850.py:354: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=(str(device) == "cuda")):



Epoch 1/3
step 30/250 loss=1.7065
step 60/250 loss=1.6402
step 90/250 loss=1.5646
step 120/250 loss=1.5397
step 150/250 loss=1.5044
step 180/250 loss=1.4769
step 210/250 loss=1.4421
step 240/250 loss=1.4126
平均 loss = 1.4067

Epoch 2/3
step 30/250 loss=1.0144
step 60/250 loss=1.0595
step 90/250 loss=1.0787
step 120/250 loss=1.0140
step 150/250 loss=1.0048
step 180/250 loss=1.0145
step 210/250 loss=0.9997
step 240/250 loss=0.9843
平均 loss = 0.9794

Epoch 3/3
step 30/250 loss=0.7290
step 60/250 loss=0.7187
step 90/250 loss=0.7180
step 120/250 loss=0.7184
step 150/250 loss=0.7205
step 180/250 loss=0.7179
step 210/250 loss=0.7077
step 240/250 loss=0.7097
平均 loss = 0.7058

正在預測 test 欄位： verification_timeline

submission 預覽：
      id promise_status  verification_timeline evidence_status  \
0  12001            Yes                already             Yes   
1  12002            Yes                already             Yes   
2  12003            Yes                already             Yes   
3  12004

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>